Ingest data to Bronze Layer

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

In [0]:
brand_schema  = StructType([
  StructField('brand_code', StringType(), False),
  StructField('brand_name', StringType(), True),
  StructField('category_code', StringType(), True)])

raw_data_path = "/Volumes/ecommerce/source_data/raw/brands/*.csv" 

df_brand = (
  spark
    .read
    .format("csv")
    .option('header', True)
    .option("delimiter", ",")
    .schema(brand_schema)
    .load(raw_data_path)
)

df_brand= df_brand.withColumn("_source_file",F.col("_metadata.file_path")) \
           .withColumn("ingested_at",F.current_timestamp())
           
df_brand.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.bronze.brz_brands")


In [0]:
category_schema = StructType([
  StructField('category_code', StringType(), False),
  StructField('category_name', StringType(), True)])

raw_data_path = "/Volumes/ecommerce/source_data/raw/category/*.csv" 

df_category = (
  spark
    .read
    .format("csv")
    .option('header', True)
    .option("delimiter", ",")
    .schema(category_schema)
    .load(raw_data_path)
)

df_category= df_category.withColumn("_source_file",F.col("_metadata.file_path")) \
           .withColumn("ingested_at",F.current_timestamp())
           
df_category.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.bronze.brz_category")


In [0]:
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),  #datatype is string due to incoming data contain anamolies
    StructField("length_cm", StringType(), True),     #datatype is string due to incoming data contain anamolies
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), False),
    StructField("ingest_timestamp", TimestampType(), False)
])

raw_data_path = "/Volumes/ecommerce/source_data/raw/products/*.csv" 

df_products = (
  spark
    .read
    .format("csv")
    .option('header', True)
    .option("delimiter", ",")
    .schema(products_schema)
    .load(raw_data_path)
)

df_products = df_products.withColumn("_source_file",F.col("_metadata.file_path")) \
           .withColumn("ingested_at",F.current_timestamp())
           
df_products.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.bronze.brz_products")

In [0]:
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

raw_data_path = "/Volumes/ecommerce/source_data/raw/customers/*.csv" 

df_customers = (
  spark
    .read
    .format("csv")
    .option('header', True)
    .option("delimiter", ",")
    .schema(customers_schema)
    .load(raw_data_path)
)

df_customers= df_customers.withColumn("_source_file",F.col("_metadata.file_path")) \
           .withColumn("ingested_at",F.current_timestamp())
           
df_customers.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.bronze.brz_customers")

In [0]:
date_schema = StructType([
    StructField("date", StringType(), True),           # Raw date in string format
    StructField("year", IntegerType(), True),          # Year
    StructField("day_name", StringType(), True),       # Day name (can be mixed case)
    StructField("quarter", IntegerType(), True),       # Quarter
    StructField("week_of_year", IntegerType(), True),  # Week of year (can be negative)
])

raw_data_path = "/Volumes/ecommerce/source_data/raw/date/*.csv" 

df_date = (
  spark
    .read
    .format("csv")
    .option('header', True)
    .option("delimiter",",")
    .schema(date_schema)
    .load(raw_data_path)
)           

df_date= df_date.withColumn("_source_file",F.col("_metadata.file_path")) \
           .withColumn("ingested_at",F.current_timestamp())
           
df_date.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.bronze.brz_date")
